# AI with Groovy

Local-model AI from Groovy notebook cells, ported from the
[Exploring AI with Groovy](https://groovy.apache.org/blog/groovy-ai) blog post
and its [companion repo](https://github.com/paulk-asert/groovy-ai):
[Ollama4j](https://github.com/ollama4j/ollama4j) (MIT) for direct Ollama chat,
then [LangChain4j](https://github.com/langchain4j/langchain4j) (Apache-2.0) for
conversational memory, structured output into Groovy records, tool calling, and
a token-streaming finale that renders into an in-place-updating display.

**Requires a local [Ollama](https://ollama.com/) server** on `localhost:11434`
with two smallish open models pulled (`ollama pull mistral:7b` and
`ollama pull qwen3:8b`) — everything runs locally: no API keys, no token bills.
This notebook is *not* runnable on Binder (the Binder image has no Ollama).

In [ ]:
@Grab('io.github.ollama4j:ollama4j:1.1.7')
@Grab('dev.langchain4j:langchain4j:1.19.0')
@Grab('dev.langchain4j:langchain4j-ollama:1.19.0')
import io.github.ollama4j.Ollama
ollama = new Ollama(requestTimeoutSeconds: 300)
"Found ollama: ${ollama.ping()}"

## Chatting with Ollama4j

A direct client for the local Ollama server. The model replies in markdown, so
`displayMarkdown` renders it nicely:

In [ ]:
import io.github.ollama4j.models.chat.OllamaChatMessageRole
import io.github.ollama4j.models.chat.OllamaChatRequest
request = OllamaChatRequest.builder().withModel('mistral:7b')
        .withMessage(OllamaChatMessageRole.USER,
                'What are 4 interesting things to do while I am on vacation in Caloundra?')
        .build()
displayMarkdown(ollama.chat(request, null).responseModel.message.response)
null

## LangChain4j: an assistant with memory

`AiServices` turns an interface into a working assistant. The assistant lives
in the session binding — so the *conversation continues across cells*:

In [ ]:
import dev.langchain4j.model.ollama.OllamaChatModel
import dev.langchain4j.memory.chat.MessageWindowChatMemory
import dev.langchain4j.service.AiServices
import java.time.Duration

interface HolidayAssistant { String chat(String message) }

model = OllamaChatModel.builder()
        .baseUrl('http://localhost:11434')
        .timeout(Duration.ofMinutes(5))
        .modelName('mistral:7b')
        .build()
assistant = AiServices.builder(HolidayAssistant)
        .chatModel(model)
        .chatMemory(MessageWindowChatMemory.withMaxMessages(10))
        .build()
displayMarkdown(assistant.chat('What are 4 interesting things to do while I am on vacation in Caloundra?'))
null

In [ ]:
// a follow-up in a new cell: the assistant remembers the conversation so far
displayMarkdown(assistant.chat('''
It might rain at some point on the weekend, so give me a very short description
of a single backup alternative if it rains. Make it different to your previous
suggestions since I am not sure which ones I will have already seen.
'''))
null

## Structured output into Groovy records

Declare a record and an interface returning it; with JSON-schema capability the
model's reply is deserialized straight into record instances — which the kernel
then renders as a table, because the result is just a `List<Map>`:

In [ ]:
import static dev.langchain4j.model.chat.Capability.RESPONSE_FORMAT_JSON_SCHEMA

record Activity(String activity, String location, String day, String time) {}
interface HolidayBot { List<Activity> extractActivitiesFrom(String text) }

bot = AiServices.builder(HolidayBot)
        .chatModel(OllamaChatModel.builder()
                .baseUrl('http://localhost:11434')
                .supportedCapabilities(RESPONSE_FORMAT_JSON_SCHEMA)
                .timeout(Duration.ofMinutes(5))
                .modelName('mistral:7b')
                .build())
        .chatMemory(MessageWindowChatMemory.withMaxMessages(10))
        .build()
activities = bot.extractActivitiesFrom('''
Suggest 4 weekend activities in Caloundra with a location, day of the weekend,
and suggested time of day for each.
''')
activities.collect { [activity: it.activity(), location: it.location(), day: it.day(), time: it.time()] }

## Tool calling

Give the model tools — plain Groovy methods with `@Tool` — and it consults them
instead of hallucinating. The weather tool below fakes a heatwave Saturday and
a rainy Sunday; `qwen3:8b` is more reliable at tool use than mistral:

In [ ]:
import dev.langchain4j.agent.tool.Tool
import java.time.LocalDate
import java.time.DayOfWeek
import java.time.temporal.TemporalAdjusters

interface ToolAssistant { String chat(String message) }
record Weather(String forecast, int minTemp, int maxTemp) {}

class HolidayTools {
    @Tool('The LocalDate of the start of the coming weekend')
    LocalDate getWeekend() { LocalDate.now().with(TemporalAdjusters.nextOrSame(DayOfWeek.SATURDAY)) }

    int fakeDay = 0
    @Tool('The expected Weather with forecast plus min and max temperature in Celsius for a given location and LocalDate')
    Weather getWeather(String location, LocalDate date) {
        def fake = [[Caloundra: new Weather('Sunny and Hot', 30, 37)],
                    [Caloundra: new Weather('Raining', 5, 15)]]
        fake[fakeDay++ % 2][location]
    }
}

toolAssistant = AiServices.builder(ToolAssistant)
        .chatModel(OllamaChatModel.builder()
                .baseUrl('http://localhost:11434')
                .timeout(Duration.ofMinutes(5))
                .modelName('qwen3:8b')
                .build())
        .chatMemory(MessageWindowChatMemory.withMaxMessages(10))
        .tools(new HolidayTools())
        .build()
displayMarkdown(toolAssistant.chat('''
Recommend an interesting thing to see in Caloundra for each day of this coming
weekend. Factor in expected weather when making recommendations. Do not
hallucinate weather or dates.
'''))
null

## Streaming into a live display

The kernel's `display`/`updateDisplay` pair is the streaming primitive: create
a display, then update it in place as tokens arrive. The handler is a plain
Groovy map coerced to the listener interface — closures see the session
variables, and `updateDisplay` is safe to call from the model's callback
thread. Watch the poem write itself:

In [ ]:
import dev.langchain4j.model.ollama.OllamaStreamingChatModel
import dev.langchain4j.model.chat.response.StreamingChatResponseHandler
import java.util.concurrent.CountDownLatch
import java.util.concurrent.TimeUnit

streamModel = OllamaStreamingChatModel.builder()
        .baseUrl('http://localhost:11434')
        .timeout(Duration.ofMinutes(5))
        .modelName('mistral:7b')
        .build()
poem = display('...')
sb = new StringBuilder()
finished = new CountDownLatch(1)
chunks = 0
streamModel.chat('Write a four-line poem about the Groovy programming language.', [
    onPartialResponse : { partial, ctx -> sb << partial.text(); chunks++; updateMarkdown(poem, sb.toString()) },
    onCompleteResponse: { response -> finished.countDown() },
    onError           : { t -> updateDisplay(poem, "stream failed: $t"); finished.countDown() }
] as StreamingChatResponseHandler)
finished.await(5, TimeUnit.MINUTES)
"streamed $chunks chunks"